In [17]:
# Импорты и загрузка данных
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

if '../src' not in sys.path:
    sys.path.append('../src')
from database import load_worst_corrosion_by_component as load_data

DF_full = load_data()

In [18]:
# Целевая переменная
TARGET = 'corr_rate_worst_mm_per_year'

assert TARGET in DF_full.columns, f'В данных отсутствует {TARGET}'

print(f"Полный датасет: {len(DF_full):,} строк, {len(DF_full.columns)} колонок")
print('Колонки:', sorted(DF_full.columns.tolist()))

Полный датасет: 36,093 строк, 62 колонок
Колонки: ['ammonia_content', 'ammonium_chloride_content', 'avg_corr_rate_mm_per_year', 'chloride_aggressiveness', 'chlorine_content', 'co2_content', 'component', 'component_id', 'component_type', 'component_type_code', 'contour', 'corr_rate_worst_mm_per_year', 'corrosion_aggressiveness_index', 'corrosion_inhibitor_content', 'corrosion_protection_index', 'cross_section_area_mm2', 'curr_measurement_date', 'curr_thickness_avg_mm', 'curr_thickness_worst_mm', 'delta_years', 'effective_start_date', 'equipment', 'equipment_age_years', 'equipment_id', 'h2s_aggressiveness_index', 'h2s_content', 'h2s_water_ratio', 'hydrochloric_acid_content', 'initial_thickness', 'inner_diameter', 'installation', 'installation_id', 'material_code', 'material_code_id', 'material_grade', 'material_resistance_score', 'material_type', 'max_corr_rate_mm_per_year', 'min_corr_rate_mm_per_year', 'nominal_eff', 'nominal_thickness_mmc', 'num_points_in_section', 'num_sections', 'out

In [13]:
# Фильтр по установкам
# 54 - АВТ-6, 68 - АВТ-2, 53 - АВТ-1, 80 - АВТ-5, 4 - KK-2, 51 - КК
INSTALLATION_IDS = [4, 51]

DF_loaded = DF_full[DF_full['installation_id'].isin(INSTALLATION_IDS)].copy()
DF_loaded = DF_loaded.sort_values('installation_id').reset_index(drop=True)
print(f"Отфильтровано (installation_id in {INSTALLATION_IDS}): {len(DF_loaded):,} строк")

# Только неотрицательная целевая переменная
DF_loaded = DF_loaded[DF_loaded[TARGET] >= 0]
print(f"После отбора строк с {TARGET} >= 0: {len(DF_loaded):,} строк")

# Пустые в числовых признаках (содержание, индексы) = 0; цель не заполняем
num_cols = [c for c in DF_loaded.select_dtypes(include=[np.number]).columns if c != TARGET]
DF_loaded[num_cols] = DF_loaded[num_cols].fillna(0)

print("Распределение по установкам:")
print(DF_loaded['installation_id'].value_counts().sort_index())

Отфильтровано (installation_id in [4, 51]): 14,480 строк
Распределение по установкам:
installation_id
4     6951
51    7529
Name: count, dtype: int64


In [19]:
# Признаки с наилучшей корреляцией с целью (по выводам correlation_analysis.ipynb)
# Рейтинг |r|: chlorine 0.231, equipment_age -0.224, nominal_thickness 0.149,
# cross_section 0.137, underdeposit -0.133, sodium_hydroxide 0.129, h2s_content 0.115
FEATURES = [
    'chlorine_content',
    'equipment_age_years',
    'nominal_thickness_mmc',
    'cross_section_area_mm2',
    'underdeposit_corrosion_index',
    'sodium_hydroxide_content',
    'h2s_content',
]
FEATURES = [c for c in FEATURES if c in DF_loaded.columns]
print('Признаки для регрессии:', FEATURES)

Признаки для регрессии: ['chlorine_content', 'equipment_age_years', 'nominal_thickness_mmc', 'cross_section_area_mm2', 'underdeposit_corrosion_index', 'sodium_hydroxide_content', 'h2s_content']


In [23]:
# Подготовка выборки (фильтр и заполнение нулями уже в DF_loaded)
from sklearn.model_selection import train_test_split

cols_use = FEATURES + [TARGET]
df_reg = DF_loaded[cols_use].copy()
df_reg = df_reg.dropna(subset=[TARGET])
n_samples = len(df_reg)
print(f"Строк с заданной целью: {n_samples:,}")
if n_samples < 10:
    raise ValueError(f"Слишком мало строк с заданной целью ({n_samples}). Проверьте колонку {TARGET}")

X = df_reg[FEATURES]
y = df_reg[TARGET]
test_size = 0.2 if n_samples >= 50 else max(0.15, 2 / n_samples)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42
)
print(f"Обучающая выборка: {len(X_train):,} строк")
print(f"Тестовая выборка:  {len(X_test):,} строк")

После заполнения признаков нулями, отбрасывания строк без цели и строк с целью < 0: 12,540 строк (из 14,480)
Обучающая выборка: 10,032 строк
Тестовая выборка:  2,508 строк


In [24]:
# Линейная регрессия
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("Метрики на обучении:")
print(f"  R²   = {r2_score(y_train, y_pred_train):.4f}")
print(f"  MAE  = {mean_absolute_error(y_train, y_pred_train):.6f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(y_train, y_pred_train)):.6f}")
print("\nМетрики на тесте:")
print(f"  R²   = {r2_score(y_test, y_pred_test):.4f}")
print(f"  MAE  = {mean_absolute_error(y_test, y_pred_test):.6f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(y_test, y_pred_test)):.6f}")

Метрики на обучении:
  R²   = 0.1219
  MAE  = 0.058996
  RMSE = 0.096624

Метрики на тесте:
  R²   = 0.0915
  MAE  = 0.057571
  RMSE = 0.090404


In [ ]:
# Коэффициенты модели
coef_df = pd.DataFrame({
    'признак': FEATURES,
    'коэффициент': model.coef_
}).sort_values('коэффициент', key=abs, ascending=False)
print(coef_df.to_string(index=False))
print(f"\nСвободный член (intercept): {model.intercept_:.6f}")